# 🏛️ Contract Analysis AI — DistilBERT Fine-Tuning (Google Colab)

This notebook trains a **DistilBERT** model on contract clause classification using the  
`alisha4walunj/quad_ledgar_merged_dataset` dataset.

**Steps:**
1. ⚙️  Install dependencies
2. 📥 Load + preprocess dataset from HuggingFace Hub
3. 🔤 Tokenize with DistilBERT tokenizer
4. 🏋️ Fine-tune with HuggingFace Trainer (accuracy + F1)
5. 💾 Save model & label map to Google Drive
6. 📦 Optionally download as `.zip` for local use

> **Runtime**: Set to **GPU → T4** via *Runtime → Change runtime type*

## 1 ⚙️  Install Dependencies

In [1]:
%%capture
!pip install -U -q transformers accelerate datasets evaluate scikit-learn accelerate peft
print('✅ Dependencies installed')

## 2 🔧 Configuration

In [2]:
import os, json, re, warnings
warnings.filterwarnings('ignore')

# ── User-configurable ──────────────────────────────────────────────
DATASET_NAME  = 'alisha4walunj/quad_ledgar_merged_dataset'
TEXT_COLUMN   = 'text'   # change if the column name differs in the dataset
LABEL_COLUMN  = 'label'
MODEL_NAME    = 'distilbert-base-uncased'
MAX_LENGTH    = 512
TEST_SIZE     = 0.15
NUM_EPOCHS    = 3
TRAIN_BATCH   = 32   # T4 can handle 32; reduce if OOM
EVAL_BATCH    = 64
LEARNING_RATE = 2e-5
OUTPUT_DIR    = '/content/contract_classifier'
LABEL_MAP_PATH = '/content/label_map.json'
# ───────────────────────────────────────────────────────────────────

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


## 3 📥 Load & Preprocess Dataset

In [3]:
from datasets import load_dataset, DatasetDict

print(f'Loading dataset: {DATASET_NAME} …')
raw = load_dataset(DATASET_NAME)

# Handle single-split or multi-split datasets
ds = raw['train'] if isinstance(raw, DatasetDict) else raw
print(f'Total examples: {len(ds)}')
print(f'Columns: {ds.column_names}')
ds[0]

Loading dataset: alisha4walunj/quad_ledgar_merged_dataset …


README.md:   0%|          | 0.00/312 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/17.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/53435 [00:00<?, ? examples/s]

Total examples: 53435
Columns: ['text', 'label']


{'text': 'Without the prior written consent of the other Party, a Party shall not at any time while this Agreement is in force and for a one-year period after termination of this Agreement either for itself or on behalf of any other company solicit, induce or cause any employee of the other Party or any Affiliated Company of this other Party who has been a representative of or employed by the other Party in connection with this Agreement to leave such employment.',
 'label': 'Employment Terms'}

In [4]:
# Inspect label distribution
from collections import Counter
label_counts = Counter(ds[LABEL_COLUMN])
print(f'Unique labels: {len(label_counts)}')
for lbl, cnt in sorted(label_counts.items(), key=lambda x: -x[1])[:20]:
    print(f'  {lbl}: {cnt}')

Unique labels: 16
  General Clauses: 16472
  Dispute Resolution: 5886
  Termination Clauses: 5295
  Governing Law and Jurisdiction: 3913
  Payment and Compensation: 3690
  Assignment and Transfer Restrictions: 3269
  Financials and Taxes: 2663
  Liability and Damages: 2425
  Employment Terms: 2359
  Commercial Terms: 1698
  Confidentiality and Non-Disclosure: 1375
  Compliance and Audit: 1280
  Risk and Insurance: 1270
  Intellectual Property & Ownership: 1002
  Performance Obligations: 514
  Regulatory Compliance: 324


In [5]:
# ── Clean text ──
def clean_text(text):
    if not isinstance(text, str): text = str(text)
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'[^\x20-\x7E]+', ' ', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip().lower()

ds = ds.map(lambda b: {TEXT_COLUMN: [clean_text(t) for t in b[TEXT_COLUMN]]},
            batched=True, desc='Cleaning text')

# ── Encode labels ──
unique_labels = sorted(set(ds[LABEL_COLUMN]))
label2id = {lbl: i for i, lbl in enumerate(unique_labels)}
id2label  = {i: lbl for lbl, i in label2id.items()}

ds = ds.map(lambda b: {'labels': [label2id[l] for l in b[LABEL_COLUMN]]},
            batched=True, desc='Encoding labels')

print(f'{len(unique_labels)} classes encoded.')

# ── Save label map ──
with open(LABEL_MAP_PATH, 'w') as f:
    json.dump({str(k): v for k, v in id2label.items()}, f, indent=2)
print(f'Label map saved → {LABEL_MAP_PATH}')

Cleaning text:   0%|          | 0/53435 [00:00<?, ? examples/s]

Encoding labels:   0%|          | 0/53435 [00:00<?, ? examples/s]

16 classes encoded.
Label map saved → /content/label_map.json


In [6]:
# ── Train / test split ──
splits = ds.train_test_split(test_size=TEST_SIZE, seed=42)
print(f"Train: {len(splits['train'])} | Test: {len(splits['test'])}")

Train: 45419 | Test: 8016


## 4 🔤 Tokenize

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch[TEXT_COLUMN], truncation=True,
                     padding='max_length', max_length=MAX_LENGTH)

tokenized = splits.map(tokenize_fn, batched=True, desc='Tokenizing')

# Drop raw text cols, keep what Trainer needs
keep = {'input_ids', 'attention_mask', 'labels'}
for split in tokenized:
    remove_cols = [c for c in tokenized[split].column_names if c not in keep]
    tokenized[split] = tokenized[split].remove_columns(remove_cols)

tokenized.set_format('torch')
print('Tokenization complete.')
print(tokenized)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing:   0%|          | 0/45419 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/8016 [00:00<?, ? examples/s]

Tokenization complete.
DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 45419
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 8016
    })
})


## 5 🏋️ Fine-Tune with HuggingFace Trainer

In [8]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)
print(f'Model loaded: {MODEL_NAME} with {len(label2id)} output classes')

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: distilbert-base-uncased with 16 output classes


In [9]:
import evaluate, numpy as np

accuracy_metric = evaluate.load('accuracy')
f1_metric       = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)['accuracy']
    f1  = f1_metric.compute(predictions=preds, references=labels, average='weighted')['f1']
    return {'accuracy': acc, 'f1': f1}

In [10]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    learning_rate=LEARNING_RATE,
    
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=100,
    fp16=(DEVICE == 'cuda'),   # Enable mixed precision on GPU
    report_to='none',
    
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('🚀 Starting training …')
trainer.train()

🚀 Starting training …


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.290528,0.288733,0.918787,0.919028
2,0.213312,0.247118,0.932385,0.932306
3,0.115788,0.244851,0.934755,0.934722


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=4260, training_loss=0.32629069968568325, metrics={'train_runtime': 1738.9951, 'train_samples_per_second': 78.354, 'train_steps_per_second': 2.45, 'total_flos': 1.8054116787142656e+16, 'train_loss': 0.32629069968568325, 'epoch': 3.0})

In [11]:
print('📊 Final evaluation …')
results = trainer.evaluate()
for k, v in results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

📊 Final evaluation …


RuntimeError: on_train_begin must be called before on_evaluate

## 6 💾 Save Model & Download

In [12]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Copy label map into model dir
import shutil
shutil.copy(LABEL_MAP_PATH, os.path.join(OUTPUT_DIR, 'label_map.json'))

print(f'✅ Model saved to {OUTPUT_DIR}')
!ls -lh {OUTPUT_DIR}

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to /content/contract_classifier
total 257M
drwxr-xr-x 2 root root 4.0K Apr  5 21:24 checkpoint-1420
drwxr-xr-x 2 root root 4.0K Apr  5 21:34 checkpoint-2840
drwxr-xr-x 2 root root 4.0K Apr  5 21:44 checkpoint-4260
-rw-r--r-- 1 root root 1.8K Apr  5 21:51 config.json
-rw-r--r-- 1 root root  548 Apr  5 21:51 label_map.json
-rw-r--r-- 1 root root 256M Apr  5 21:51 model.safetensors
-rw-r--r-- 1 root root  322 Apr  5 21:51 tokenizer_config.json
-rw-r--r-- 1 root root 695K Apr  5 21:51 tokenizer.json
-rw-r--r-- 1 root root 5.1K Apr  5 21:51 training_args.bin


In [ ]:
# ── Option A: Save to Google Drive ──────────────────────────────────
# Uncomment to mount Drive and copy the model folder there.

# from google.colab import drive
# drive.mount('/content/drive')
# drive_path = '/content/drive/MyDrive/contract_classifier'
# shutil.copytree(OUTPUT_DIR, drive_path, dirs_exist_ok=True)
# print(f'Model also saved to Google Drive: {drive_path}')

In [13]:
# ── Option B: Download as zip ────────────────────────────────────────
# Creates a zip and triggers browser download.

import shutil
from google.colab import files

zip_path = '/content/contract_classifier.zip'
shutil.make_archive('/content/contract_classifier', 'zip', OUTPUT_DIR)
print(f'Zip created: {zip_path}')
files.download(zip_path)

Zip created: /content/contract_classifier.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ✅ Next Steps — Plug Into Local FastAPI Backend

After downloading `contract_classifier.zip`:

```bash
# Unzip into your models directory
Expand-Archive contract_classifier.zip -DestinationPath contract-ai/models/contract_classifier

# Copy label map
copy contract-ai/models/contract_classifier/label_map.json contract-ai/models/label_map.json

# Start the API
cd contract-ai
uv run uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload
```

Then call:
```bash
curl -X POST http://localhost:8000/api/v1/predict \
     -H 'Content-Type: application/json' \
     -d '{"text": "This agreement may be terminated by either party upon 30 days written notice."}'
```